This notebook aim to help getting started with ``xLLiM`` library. It presents a complete and detailed workflow with a simple example. The workflow can be summed up into 3 main section:
 1. Generate a training dataset (optional)
 2. Train your GLLiM model
 3. Prediction
 4. Apply importance sampling (optional)
 5. Post treatment analysis

#### Import xLLiM module

In [ ]:
import xllim
import numpy as np
import matplotlib.pyplot as plt
import json
import logging
logging.getLogger().setLevel(logging.INFO)

## 1. Generate a training Dataset

=> If you have a ready-to-use Dataset, go to next section. 

Otherwise, you can generate a training dataset with xllim library from physical models. For now there a two concrete Photometric models implemented (Hapke and Shkuratov), one very basic Test model and a custom class (ExternalPythonModel) where you can define your own functional model in Python language.
More details are available in [documentation](https://kernelo-mistis.gitlabpages.inria.fr/kernelo-gllim-is/functional_model.html)


#### Create functional models : the Photometric Hapke model

In the example we create an Hapke photometric model. It requires a matrix of geometries that will be used by the model. The shape of the Numpy array should be *(n_geometries,3)*. The three geometric angles must be this particular order :
 - At [:,0] the incidence angle (inc) also called the solar zenith angle (sza)
 - At [:,1] the emergence angle (eme) also called the vertical zenith angle (vza)
 - At [:,2] the phi angle (phi)

In [ ]:
# Get geometries from JSON file
with open("JSC1_BRDF.json", "r") as f:
    data = json.load(f)
geometries = np.array(data["JSC1_analogue"]["geometries"])

print(geometries)
print(geometries.shape)


# Define other Hapke arguments
variant = "2002"
adapter = "six"
theta_bar_scaling = 30
b0 = 0
h = 0.1

# Create Hapke model
model = xllim.HapkeModel(geometries, variant, adapter, theta_bar_scaling, b0, h)

You can use your model to apply basic class methods :
 - **F** : the functional model F describing the functional model.
 - **getDimensionY** : returns the dimension of Y
 - **getDimensionX** : return de dimension of X
 - **toPhysic** : converts the X data from mathematical framework (0<X<1) to physical framework
 - **fromPhysic** : converts the X data from physical framework to mathematical framework (0<X<1)

In [ ]:
L = model.getDimensionX()
D = model.getDimensionY()
print("Test model transform a vector X of dimension L = {} into a vector Y of dimension D = {}\n".format(L, D))

x = np.ones(L)
x_physic = model.toPhysic(x)
x_mat = model.fromPhysic(x_physic)
print("model.toPhysic({}) = {}".format(x, x_physic))
print("model.fromPhysic({}) = {}\n".format(x_physic, x_mat))

x = np.ones(L)
y = model.F(x)
print("model.F({}) = ".format(x))
print(y)


#### Generate a dataset from the functional model

The dataset is composed of 
 - x_gen with shape (N, L)
 - y_gen with shape (N, D)

In [ ]:
N = 10_000                                          # number of generated observation
generator_type = "sobol"                            # the type of the generator used to generate x_gen matrix values
covariance = np.ones(model.getDimensionY()) * 1e-5  # vector of dimension D coresponding to the y_i variances.
seed = 12345                                        # seed number for random generators

x_gen, y_gen = model.genData(N, generator_type, covariance, seed)

print(x_gen.shape)
print(y_gen.shape)

## 2. Train your GLLiM model

##### Instanciate a GLLiM object
You can instanciate a GLLiM object according with constraints.
Refer to the documentation for more details on the notation and the mathematical aspect

In [ ]:
# Choose Covariance matrix type
gamma_type = "Full"
sigma_type = "Diag"

# Define model dimensions
# ! D and L have to fit the dataset dimensions
K = 5              # number of gaussians in the mixture model
D = y_gen.shape[1]  # = model.getDimensionY() = 9 (TestModel)
L = x_gen.shape[1]  # = model.getDimensionX() = 4 (TestModel)

# Hybrid model
n_hidden_variables = 0
L_t = L - n_hidden_variables
L_w = n_hidden_variables


gllim = xllim.GLLiM(
    K, D, L, gamma_type="full", sigma_type="diag", n_hidden_variables=n_hidden_variables
)

You can set an intial theta to your model before the initialisation and/or the training step if you want. Theta is initialized with default settings as, $\forall$ k $\in$ [0,K-1] :
 - Pi[k] = 1/K
 - A[k] = $O_{D,L}$
 - B[k] = $O_{D}$
 - C[k] = $O_{L}$
 - Gamma[k] = $I_{L,L}$
 - Sigma[k] = $I_{D,D}$

In [ ]:
# print(gllim.getParams())
# print(gllim.getParamPi())
# print(gllim.getParamA())
print(gllim.getParamC())
# print(gllim.getParamGamma())
# print(gllim.getParamB())
# print(gllim.getParamSigma())

gllim.setParamC(np.ones((K, L)) * 0.02)

print(gllim.getParamC())

Apply GLLiM initialisation step

In [ ]:
# initialisation parameters
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 10
gmm_floor = 1e-12
nb_experiences = 5

seed = 12345678
verbose = 1

# TODO : transpose in bindings & CARMA "can't be copied or stolen" : unable copying
x_gen_T = np.array(x_gen.T)
y_gen_T = np.array(y_gen.T)

gllim.initialize(
    x_gen_T,
    y_gen_T,
    gllim_em_iteration,
    gllim_em_floor,
    gmm_kmeans_iteration,
    gmm_em_iteration,
    gmm_floor,
    nb_experiences,
    seed,
    verbose,
)

Train GLLiM model with GLLiM-EM algorithm

In [ ]:
# training parameters
train_max_iteration = 100
train_ratio_ll = 1e-5
train_floor = 1e-12

gllim.train(
    x_gen_T, y_gen_T, train_max_iteration, train_ratio_ll, train_floor, verbose
)


Once trained you can retrieve the inverted model parameters $\theta^*$

In [ ]:
theta_star = gllim.getInverse()
print(theta_star.Pi)

## 3. Prediction

You can use your trained GLLiM in order to evaluate probability densites :
 - directDensities(x) correspond to the **forward conditional density** and returns $E[Y = y|X = x; \theta]$
 - inverseDensities(x) correspond to the **backward** or **inverse conditional density** and returns $E[Y = y|X = x; \theta]$

In the case your dataset is represented by a *functional model*, GLLiM model is your surrogate model analytically inversible. You can thus evaluate very quickly the inverted F model on a large number of observed Y :

$x = F^{-1}(y) \approx GLLiM^{-1}(y)$

#### Get observations Y corresponding to mesured reflectance

In [ ]:
# Get geometries from JSON file
with open("JSC1_BRDF.json", "r") as f:
    data = json.load(f)
wavelengths = np.array(data["JSC1_analogue"]["wavelengths"])
reflectances = np.array(data["JSC1_analogue"]["reflectance_JSC1"])
N_obs = wavelengths.shape[0]

print("There is {} observations !".format(N_obs))
print("wavelengths has shape : {}".format(wavelengths.shape))
print("reflectances has shape : {}".format(reflectances.shape))

observations = np.array(reflectances[:,0,:].T)
incertitudes = np.array(reflectances[:,1,:].T)

print("observations has shape : {}".format(observations.shape))
print("incertitudes has shape : {}".format(incertitudes.shape))


### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 3, figsize=(30, 5))
fig.suptitle("Reflectances", fontsize=16)
y_idx_list = [3, 45, 68]
for d in range(len(y_idx_list)):
    axs[d].plot(wavelengths, observations[y_idx_list[d]], 'b-')
    axs[d].set_xlabel("Observations")
    axs[d].set_title('Y_'+ str(y_idx_list[d]))
    
plt.show()


Then we apprly the inversion

In [ ]:
predictions = gllim.inverseDensities(observations, incertitudes)

# comparing to y_obs
print("y_obs[0] = {}".format(observations[:,0]))
print("y_predicted[0] = F(x_predicted[0]) = {}".format(model.F(predictions.meanPredResult.mean[0])))

## 4. Importance sampling

The sampling step depends on a *functional model*. If our case we try to improve our result thank to the IMIS (Incremental Mixture Importance Sampling) algorithm 

In [ ]:
# Firstly the information of the GMMs of the proposition law is retrieve from the prediction results
proposition_gmms = []
for i in range(N_obs):
    gmm_weight = np.array(predictions.meanPredResult.gmm_weights[i])
    gmm_mean = np.array(predictions.meanPredResult.gmm_means[i])
    gmm_cov = np.array(predictions.meanPredResult.gmm_covs)
    proposition_gmms.append((gmm_weight, gmm_mean, gmm_cov))

# Covariance on the target density
covariance = np.ones(model.getDimensionY()) * 0.001

# Importance sampling parameters
N_0 = 1000
B = 50
J = 20

# TODO : transpose in bindings & CARMA "can't be copied or stolen" : unable copying
y_obs_noised_T = np.array(observations.T)
y_obs_noise_T = np.array(incertitudes.T)

is_results = model.importanceSampling(proposition_gmms, y_obs_noised_T, y_obs_noise_T, covariance, N_0, B, J, 0)

In [ ]:
# comparing to x_obs
print("y_obs[0] = {}".format(observations[:,0]))
print("y_predicted[0] = {}".format(model.F(predictions.meanPredResult.mean[0])))
print("y_is[0] = {}".format(model.F(is_results.predictions[:,0])))

## 5. Post-processing analysis

#### Evaluate reconstruction error

In [ ]:
def compute_reconstruction_error(reconstruction, observation):
    return np.linalg.norm(observation - reconstruction) / np.linalg.norm(observation)

reconstruction_error = []
reconstruction_error_is = []
for n in range(N_obs):
    reconstruction_error.append(
        compute_reconstruction_error(model.F(predictions.meanPredResult.mean[n]), observations[:,n])
    )
    reconstruction_error_is.append(
        compute_reconstruction_error(model.F(is_results.predictions[:,n]), observations[:,n])
    )
    

plt.figure(figsize=(20, 5))
plt.plot(wavelengths, reconstruction_error, 'b*', label='GLLiM prediction')
plt.plot(wavelengths, reconstruction_error_is, 'g+', label='GLLiM + IS')
plt.xlabel("Observations")
plt.title("Reconstruction error for each observation")
plt.legend()
    
plt.show()

#### Comparing estimation methods (GLLiM prediction and GLLiM + IS prediction)

In [ ]:
### Plot results ###
fig, axs = plt.subplots(2, L//2, figsize=(20, 10))
fig.suptitle("X Parameters estimations", fontsize=16)

for l in range(L):
    i = 0 if (l*2 < L) else 1
    j = l%(L//2)
    axs[i,j].plot(wavelengths, predictions.meanPredResult.mean[:,l], 'b-', label='GLLiM prediction')
    axs[i,j].plot(wavelengths, is_results.predictions[l], 'g-', label='GLLiM + IS')
    axs[i,j].set_xlabel("Observations")
    axs[i,j].set_title('X_'+ str(l))
    axs[i,j].legend()
    
plt.show()